**Part 1: Option A**

In [8]:
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests

In [ ]:
url = "https://tastesbetterfromscratch.com/meal-plan-194/"
response = requests.get(url, impersonate="chrome")

200
https://tastesbetterfromscratch.com/meal-plan-194/
<!DOCTYPE html><html lang="en-US">
<head><meta charset="UTF-8"><script>if(navigator.userAgent.match(/MSIE|Internet Explorer/i)||navigator.userAgent.match(/Trident\/7\..*?rv:11/i)){var href=document.lo


In [69]:
html = response.text
soup = BeautifulSoup(html, "html.parser")

In [17]:
meal_rows = soup.find_all("p", class_="has-text-align-left")
first_meal = meal_rows[0]
print(first_meal.prettify())

<p class="has-text-align-left wp-block-paragraph">
 <strong>
  Monday
 </strong>
 :
 <a href="https://tastesbetterfromscratch.com/ground-beef-tacos/">
  Ground Beef Tacos
 </a>
 $11.15
</p>



In [ ]:
day = first_meal.find("strong").get_text(strip=True)
recipe_name = first_meal.find("a").get_text(strip=True)
recipe_link = first_meal.find("a")["href"]
recipe_price = first_meal.get_text(" ", strip =True).split("$")[-1]
recipe_price = float(recipe_price)

(day, recipe_name, recipe_link, recipe_price) 

('Monday',
 'Ground Beef Tacos',
 'https://tastesbetterfromscratch.com/ground-beef-tacos/',
 11.15)

In [ ]:
monday_dictionary = {
    "Day of the Week": day,
    "Name of Recipe": recipe_name,
    "Link to Recipe": recipe_link,
    "Price of Recipe": recipe_price
}

monday_dictionary

{'Day of the Week': 'Monday',
 'Name of Recipe': 'Ground Beef Tacos',
 'Link to Recipe': 'https://tastesbetterfromscratch.com/ground-beef-tacos/',
 'Price of Recipe': 11.15}

In [72]:
records = []

for meal in meal_rows:
    day = meal.find("strong").get_text(strip=True).rstrip(":")
    recipe_name = meal.find("a").get_text(strip=True)
    recipe_link = meal.find("a")["href"]
    recipe_price = meal.get_text(" ", strip =True).split("$")[-1]
    recipe_price = float(recipe_price)
    records.append({
        "Day of the Week": day,
        "Name of Recipe": recipe_name,
        "Link to Recipe": recipe_link,
        "Price of Recipe": recipe_price
    })

records

df = pd.DataFrame(records)
df

,Day of the Week,Name of Recipe,Link to Recipe,Price of Recipe
0,Monday,Ground Beef Tacos,https://tastesbetterfromscratch.com/ground-bee...,11.15
1,Tuesday,Chow Mein,https://tastesbetterfromscratch.com/chinese-ch...,10.36
2,Wednesday,Patty Melt,https://tastesbetterfromscratch.com/patty-melt/,18.00
3,Thursday,Chicken and Dumplings,https://tastesbetterfromscratch.com/chicken-an...,14.56
4,Friday,Veggie Wrap,https://tastesbetterfromscratch.com/hummus-eda...,9.79


**Part 2: Option A**

In [85]:
monday_recipe = df.loc[0, "Name of Recipe"]
monday_recipe

'Ground Beef Tacos'

In [93]:
search_term = "tacos"

domain = "https://tasty.p.rapidapi.com"
endpoint = "recipes/list"
api_url = f"{domain}/{endpoint}"

headers = {
    "X-RapidAPI-Key": "98a1a38689msh828b02a57ecf516p106f60jsncd26bb67b1fa",
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

params = {
    "from": 0,
    "size": 40,
    "q": search_term
}

In [94]:
api_response = requests.get(
    api_url,
    headers=headers,
    params=params
)

In [95]:
tasty_data = api_response.json()

print(tasty_data.keys())
print("Total matches:", tasty_data["count"])
print("Recipes returned:", len(tasty_data["results"]))

dict_keys(['count', 'results', 'aggregations'])
Total matches: 267
Recipes returned: 40


In [ ]:
tasty_recipes = tasty_data["results"]
first_tasty_recipe = tasty_recipes[0]
print(first_tasty_recipe.keys())

dict_keys(['nutrition_visibility', 'country', 'instructions', 'keywords', 'facebook_posts', 'language', 'seo_path', 'user_ratings', 'price', 'id', 'brand', 'slug', 'tips_summary', 'servings_noun_singular', 'show_id', 'prep_time_minutes', 'is_subscriber_content', 'sections', 'brand_id', 'tags', 'nutrition', 'is_app_only', 'name', 'compilations', 'num_servings', 'buzz_id', 'tips_and_ratings_enabled', 'aspect_ratio', 'show', 'created_at', 'description', 'draft_status', 'inspired_by_url', 'thumbnail_url', 'thumbnail_alt_text', 'total_time_minutes', 'video_url', 'updated_at', 'credits', 'approved_at', 'is_one_top', 'servings_noun_plural', 'renditions', 'beauty_url', 'is_shoppable', 'topics', 'total_time_tier', 'video_ad_content', 'seo_title', 'yields', 'original_video_url', 'canonical_id', 'cook_time_minutes', 'promotion', 'video_id'])


In [110]:
all_recipes = tasty_recipes.copy()

for start in [40, 80]:
    page_size = min(40, 100 - start)

    page_params = {
        "from": start,
        "size": page_size,
        "q": search_term
    }

    page_response = requests.get(
        api_url,
        headers=headers,
        params=page_params
    )

    page_response.raise_for_status()

    page_recipes = page_response.json()["results"]
    all_recipes.extend(page_recipes)

print(len(all_recipes))

100


In [114]:
tasty_df_full = pd.json_normalize(all_recipes)
tasty_df = tasty_df_full[[
    "id", "name", "prep_time_minutes", "total_time_minutes", "num_servings", "user_ratings.score", "nutrition.calories", "thumbnail_url"
]].copy()

tasty_df.head(10)

,id,name,prep_time_minutes,total_time_minutes,num_servings,user_ratings.score,nutrition.calories,thumbnail_url
0,3136,Taco Soup,10,30,6,0.978220,604.0,https://img.buzzfeed.com/thumbnailer-prod-us-e...
1,560,Easy Fish Tacos,15,31,8,0.974212,277.0,https://img.buzzfeed.com/video-api-prod/assets...
2,339,Taco-Stuffed Peppers,30,65,6,0.981353,541.0,https://img.buzzfeed.com/video-api-prod/assets...
3,4500,Crispy Fish Tacos,20,70,6,0.969150,380.0,https://img.buzzfeed.com/thumbnailer-prod-us-e...
4,6221,Taco Lasagna,40,70,8,0.963816,455.0,https://img.buzzfeed.com/thumbnailer-prod-us-e...
5,814,One-Pot Taco Spaghetti,10,30,4,0.943320,780.0,https://img.buzzfeed.com/video-api-prod/assets...
6,4835,Vampire Tacos,0,0,3,0.959660,NaN,https://img.buzzfeed.com/thumbnailer-prod-us-e...
7,468,Crispy Potato Tacos,20,50,8,0.911800,155.0,https://img.buzzfeed.com/video-api-prod/assets...
8,1177,Taco Pizza,15,30,4,0.958537,979.0,https://img.buzzfeed.com/video-api-prod/assets...
9,5452,Chicken Katsu Tacos,15,30,4,0.957540,NaN,https://img.buzzfeed.com/thumbnailer-prod-us-e...
